# Grapedit Cloud Backend (Colab)
Run this notebook to start a powerful free cloud backend for rendering your videos!

In [ ]:
!pip install fastapi uvicorn python-multipart nest-asyncio pyngrok

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok

# Put your ngrok auth token here if you have one, or just leave it blank for temporary sessions
NGROK_AUTH_TOKEN = ""
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print("="*60)
print(f"YOUR CLOUD RENDER URL IS: {public_url}")
print("COPY AND PASTE THIS INTO GRAPEDIT SETTINGS!")
print("="*60)

In [ ]:
from fastapi import FastAPI, UploadFile, File, Request
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import os
import uuid
import subprocess
import uvicorn

os.makedirs("/tmp/uploads", exist_ok=True)
os.makedirs("/tmp/rendered", exist_ok=True)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def read_root(): return {"status": "Running"}

@app.post("/upload")
async def upload_video(video: UploadFile = File(...)):
    filename = f"{uuid.uuid4()}{os.path.splitext(video.filename)[1]}"
    filepath = os.path.join("/tmp/uploads", filename)
    with open(filepath, "wb") as f:
        f.write(await video.read())
    return {"success": True, "filename": filename, "url": f"{public_url}/uploads/{filename}"}

@app.get("/uploads/{filename}")
def get_upload(filename: str): return FileResponse(os.path.join("/tmp/uploads", filename))

@app.get("/rendered/{filename}")
def get_rendered(filename: str): return FileResponse(os.path.join("/tmp/rendered", filename))

@app.post("/render")
async def render_video(request: Request):
    data = await request.json()
    job_id = str(uuid.uuid4())
    out_path = os.path.join("/tmp/rendered", f"{data.get('exportName', 'export')}_{job_id}.mp4")
    list_path = os.path.join("/tmp/rendered", f"list_{job_id}.txt")
    
    try:
        with open(list_path, "w") as f:
            for i, seg in enumerate(data.get("segments", [])):
                src = os.path.join("/tmp/uploads", seg["sourceFile"])
                tmp = os.path.join("/tmp/rendered", f"seg_{job_id}_{i}.mp4")
                subprocess.run(["ffmpeg", "-y", "-ss", str(seg["sourceStart"]), "-i", src, "-t", str(seg["duration"]), "-c", "copy", tmp], check=True)
                f.write(f"file '{tmp}'\n")
        subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_path, "-c", "copy", out_path], check=True)
        return {"success": True, "url": f"{public_url}/rendered/{os.path.basename(out_path)}"}
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

uvicorn.run(app, host="0.0.0.0", port=8000)